In [1]:
import os
# torchをインポートする前に設定
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
from pathlib import Path
import torch
from lerobot.configs import FeatureType
from lerobot.datasets import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.policies import make_pre_post_processors
from lerobot.policies.act import ACTConfig, ACTPolicy
from lerobot.utils.feature_utils import dataset_to_policy_features

cont_f=False


In [3]:
def make_delta_timestamps(delta_indices: list[int] | None, fps: int) -> list[float]:
    if delta_indices is None:
        return [0]
    return [i / fps for i in delta_indices]


In [4]:
output_directory = Path("output/robot_learning_tutorial/act")
output_directory.mkdir(parents=True, exist_ok=True)

# Select your device
#device = torch.device("mps")  # or "cuda" or "cpu"
device = torch.device("cuda")  # or "cuda" or "cpu"

#dataset_id = "lerobot/aloha_sim_transfer_cube_human"
dataset_id = "lerobot/svla_so101_pickplace"


# This specifies the inputs the model will be expecting and the outputs it will produce
dataset_metadata = LeRobotDatasetMetadata(dataset_id)
features = dataset_to_policy_features(dataset_metadata.features)

output_features = {key: ft for key, ft in features.items() if ft.type is FeatureType.ACTION}
input_features = {key: ft for key, ft in features.items() if key not in output_features}

cfg = ACTConfig(input_features=input_features, output_features=output_features)
#print('cfg:',cfg)
#cfg: ACTConfig(n_obs_steps=1,
# input_features={'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(6,)), 
#   'observation.images.up': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 480, 640)), 
#   'observation.images.side': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 480, 640))}, 
# output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(6,))},
# device='cuda',
# use_amp=False, 
# use_peft=False, 
# push_to_hub=True, 
# repo_id=None, 
# private=None, 
# tags=None, 
# license=None, 
# pretrained_path=None, 
# chunk_size=100, 
# n_action_steps=100, 
# normalization_mapping={'VISUAL': <NormalizationMode.MEAN_STD: 'MEAN_STD'>, 'STATE': <NormalizationMode.MEAN_STD: 'MEAN_STD'>, 'ACTION': <NormalizationMode.MEAN_STD: 'MEAN_STD'>}, 
# vision_backbone='resnet18', 
# pretrained_backbone_weights='ResNet18_Weights.IMAGENET1K_V1', 
# replace_final_stride_with_dilation=False, 
# pre_norm=False, 
# dim_model=512, 
# n_heads=8, 
# dim_feedforward=3200, 
# feedforward_activation='relu', 
# n_encoder_layers=4, 
# n_decoder_layers=1, 
# use_vae=True, 
# latent_dim=32, 
# n_vae_encoder_layers=4, 
# temporal_ensemble_coeff=None, 
# dropout=0.1, 
# kl_weight=10.0, 
# optimizer_lr=1e-05, 
# optimizer_weight_decay=0.0001, 
# optimizer_lr_backbone=1e-05)

if cont_f:
    model_id = "output/robot_learning_tutorial/act"
    policy = ACTPolicy.from_pretrained(
        model_id,
        local_files_only=True
    ).to(device)
else:
    policy = ACTPolicy(cfg)
preprocessor, postprocessor = make_pre_post_processors(cfg, dataset_stats=dataset_metadata.stats)

policy.train()
policy.to(device)
# To perform action chunking, ACT expects a given number of actions as targets
delta_timestamps = {
    "action": make_delta_timestamps(cfg.action_delta_indices, dataset_metadata.fps),
}
# add image features if they are present
delta_timestamps |= {
    k: make_delta_timestamps(cfg.observation_delta_indices, dataset_metadata.fps)
    for k in cfg.image_features
}
# Instantiate the dataset
dataset = LeRobotDataset(dataset_id, delta_timestamps=delta_timestamps)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
# Create the optimizer and dataloader for offline training
optimizer = cfg.get_optimizer_preset().build(policy.parameters())
#print('optimizer:',optimizer)
#optimizer: AdamW (
# Parameter Group 0
# amsgrad: False
# betas: (0.9, 0.999)
#capturable: False
#decoupled_weight_decay: True
#differentiable: False
#eps: 1e-08
#foreach: None
#fused: None
#lr: 1e-05
#maximize: False
#weight_decay: 0.0001
#)


#batch_size = 32
#batch_size = 8
batch_size = 4

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=device.type != "cpu",
    drop_last=True,
    num_workers=0
)

In [9]:
# Number of training steps and logging frequency
training_steps = 1
#training_steps = 300
#training_steps = 3000
#training_steps = 10000
#training_steps = 15000
#training_steps = 25000   # batch=8
#log_freq = 1
log_freq = 10
#log_freq = 100
log_freq = 200

# Run training loop
step = 0
done = False
while not done:
    for batch in dataloader:
        batch = preprocessor(batch)

        print('type(batch):',type(batch))
        # type(batch): <class 'dict'>
        #print('batch.keys():',batch.keys())
        #batch.keys(): dict_keys(['action', 'next.reward', 
        # 'next.done', 'next.truncated', 'info', 
        # 'action_is_pad', 'observation.images.up_is_pad',
        # 'observation.images.side_is_pad', 
        # 'task', 'index', 'task_index', 
        # 'episode_index', 'observation.images.up', 'observation.images.side', 
        # 'observation.state'])
        
        #print('batch[action_is_pad]:',batch["action_is_pad"])

        torch.cuda.empty_cache()

        loss, _ = policy.forward(batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if step % log_freq == 0:
            print(f"step: {step} loss: {loss.item():.3f}")
        step += 1
        if step >= training_steps:
            done = True
            break


type(batch): <class 'dict'>
step: 0 loss: 3.653


In [10]:
# Save the policy checkpoint, alongside the pre/post processors
policy.save_pretrained(output_directory)
preprocessor.save_pretrained(output_directory)
postprocessor.save_pretrained(output_directory)
# Save all assets to the Hub
#policy.push_to_hub("<user>/robot_learning_tutorial_act")
#preprocessor.push_to_hub("<user>/robot_learning_tutorial_act")
#postprocessor.push_to_hub("<user>/robot_learning_tutorial_act")